# Data Quality Review

All checks in this notebook are read-only diagnostics. Findings are logged for review; no records or values are corrected, removed, or imputed.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent
if not (ROOT / 'data' / 'raw' / 'wind_turbine_detection.csv').exists():
    ROOT = Path.cwd() / 'PROJECT 1'
DATA_PATH = ROOT / 'data' / 'raw' / 'wind_turbine_detection.csv'
df = pd.read_csv(DATA_PATH)
display(Markdown(f'Loaded raw file `{DATA_PATH.name}` with **{df.shape[0]:,} rows**.'))

Loaded raw file `wind_turbine_detection.csv` with **131,760 rows**.

In [2]:
# Structural checks
display(Markdown('## Structural data-quality checks'))
structure = pd.DataFrame({'metric': ['row_count', 'column_count', 'exact_duplicate_rows'], 'value': [len(df), df.shape[1], df.duplicated().sum()]})
display(structure)
missing = pd.DataFrame({'column': df.columns, 'data_type': df.dtypes.astype(str).values, 'missing_count': df.isna().sum().values})
missing['missing_percentage'] = missing.missing_count.div(len(df)).mul(100)
display(Markdown('### Missing values and data types'))
display(missing.sort_values(['missing_count', 'column'], ascending=[False, True]).style.format({'missing_percentage': '{:.3f}%'}))
display(Markdown(f'**Finding:** {int(df.duplicated().sum()):,} exact duplicate rows were found. **{int((missing.missing_count > 0).sum())}** columns contain missing values.'))

## Structural data-quality checks

,metric,value
0,row_count,131760
1,column_count,35
2,exact_duplicate_rows,0


### Missing values and data types

,column,data_type,missing_count,missing_percentage
16,generator_bearing_temp_C,float64,690,0.524%
14,gearbox_oil_temp_C,float64,645,0.490%
27,oil_pressure_bar,float64,636,0.483%
6,air_density_kgm3,float64,0,0.000%
7,ambient_temp_C,float64,0,0.000%
12,blade_pitch_angle_deg,float64,0,0.000%
33,component_age_days,float64,0,0.000%
29,cumulative_energy_MWh,float64,0,0.000%
20,drivetrain_vibration_rms_mmps,float64,0,0.000%
34,failure,int64,0,0.000%


**Finding:** 0 exact duplicate rows were found. **3** columns contain missing values.

In [3]:
# Timestamp quality checks
display(Markdown('## Timestamp quality checks'))
parsed_timestamp = pd.to_datetime(df['timestamp'], format='%m/%d/%Y %H:%M', errors='coerce')
timestamp_quality = df[['turbine_id', 'timestamp']].copy()
timestamp_quality['parsed_timestamp'] = parsed_timestamp
invalid_timestamps = timestamp_quality.parsed_timestamp.isna().sum()
duplicate_within_turbine = timestamp_quality.duplicated(['turbine_id', 'parsed_timestamp'], keep=False) & timestamp_quality.parsed_timestamp.notna()
# Chronology is assessed in raw-file order, independently of any later sorting.
raw_order_diff = timestamp_quality.groupby('turbine_id')['parsed_timestamp'].diff()
out_of_order = raw_order_diff.lt(pd.Timedelta(0)).sum()

ordered = timestamp_quality.dropna(subset=['parsed_timestamp']).sort_values(['turbine_id', 'parsed_timestamp'])
ordered['interval'] = ordered.groupby('turbine_id')['parsed_timestamp'].diff()
expected_interval = pd.Timedelta(minutes=10)
intervals = ordered.interval.dropna()
interval_distribution = intervals.value_counts().sort_index().rename_axis('interval').reset_index(name='occurrences')
interval_distribution['percentage'] = interval_distribution.occurrences.div(len(intervals)).mul(100)
unexpected_intervals = intervals.ne(expected_interval).sum()
gaps_over_expected = intervals.gt(expected_interval).sum()

timestamp_summary = pd.DataFrame({'check': ['invalid_or_unparseable_timestamps', 'duplicate_timestamps_within_turbine', 'out_of_order_records_in_raw_file', 'intervals_not_equal_to_expected_10_minutes', 'gaps_greater_than_10_minutes'], 'count': [invalid_timestamps, duplicate_within_turbine.sum(), out_of_order, unexpected_intervals, gaps_over_expected]})
display(timestamp_summary)
display(Markdown('### Observed intervals within turbine (after timestamp sorting)'))
display(interval_distribution.style.format({'percentage': '{:.3f}%'}))
display(Markdown(f'**Finding:** The expected SCADA interval is 10 minutes. The observed interval distribution above should be reviewed carefully; **{gaps_over_expected:,}** intervals exceed 10 minutes.'))

## Timestamp quality checks

,check,count
0,invalid_or_unparseable_timestamps,0
1,duplicate_timestamps_within_turbine,0
2,out_of_order_records_in_raw_file,0
3,intervals_not_equal_to_expected_10_minutes,0
4,gaps_greater_than_10_minutes,0


### Observed intervals within turbine (after timestamp sorting)

,interval,occurrences,percentage
0,0 days 00:10:00,131745,100.000%


**Finding:** The expected SCADA interval is 10 minutes. The observed interval distribution above should be reviewed carefully; **0** intervals exceed 10 minutes.

In [4]:
# Turbine identifier quality checks
display(Markdown('## Turbine identifier checks'))
turbine_summary = df.groupby('turbine_id', dropna=False)['failure'].agg(observations='size', normal_records=lambda x: (x == 0).sum(), fault_records=lambda x: (x == 1).sum())
turbine_summary['failure_rate_pct'] = turbine_summary.fault_records.div(turbine_summary.observations).mul(100)
median_observations = turbine_summary.observations.median()
turbine_summary['observation_deviation_from_median_pct'] = turbine_summary.observations.div(median_observations).sub(1).mul(100)
substantially_different = turbine_summary[turbine_summary.observation_deviation_from_median_pct.abs() > 10]
no_target_class = turbine_summary[(turbine_summary.normal_records == 0) | (turbine_summary.fault_records == 0)]
display(pd.DataFrame({'unique_turbines': [df.turbine_id.nunique(dropna=True)], 'missing_turbine_id_records': [df.turbine_id.isna().sum()], 'median_observations_per_turbine': [median_observations]}))
display(turbine_summary.style.format({'failure_rate_pct': '{:.3f}%', 'observation_deviation_from_median_pct': '{:+.2f}%'}))
display(Markdown('### Turbines with >10% observation-count deviation from the median'))
display(substantially_different if not substantially_different.empty else pd.DataFrame({'finding': ['None']}))
display(Markdown('### Turbines with no records for one target class'))
display(no_target_class if not no_target_class.empty else pd.DataFrame({'finding': ['None']}))

## Turbine identifier checks

,unique_turbines,missing_turbine_id_records,median_observations_per_turbine
0,15,0,8784.0


,observations,normal_records,fault_records,failure_rate_pct,observation_deviation_from_median_pct
turbine_id,,,,,
T001,8784,8226,558,6.352%,+0.00%
T002,8784,8615,169,1.924%,+0.00%
T003,8784,8618,166,1.890%,+0.00%
T004,8784,8429,355,4.041%,+0.00%
T005,8784,8544,240,2.732%,+0.00%
T006,8784,8503,281,3.199%,+0.00%
T007,8784,8292,492,5.601%,+0.00%
T008,8784,8306,478,5.442%,+0.00%
T009,8784,8684,100,1.138%,+0.00%


### Turbines with >10% observation-count deviation from the median

,finding
0,None


### Turbines with no records for one target class

,finding
0,None


In [5]:
# Engineering-context anomaly screens. These are review flags, not fault rules.
display(Markdown('## SCADA anomaly screens'))
checks = [
    ('negative_power_output', df.power_output_kW < 0, 'Power output below zero is physically implausible.'),
    ('power_above_rated_power', df.power_output_kW > df.rated_power_kW, 'Output above the recorded rated capacity warrants review.'),
    ('negative_rotor_speed', df.rotor_speed_rpm < 0, 'Negative rotational speed is implausible.'),
    ('negative_generator_speed', df.generator_speed_rpm < 0, 'Negative rotational speed is implausible.'),
    ('wind_speed_negative_or_above_40_mps', (df.wind_speed_mps < 0) | (df.wind_speed_mps > 40), 'Negative or exceptionally high wind-speed reading.'),
    ('wind_direction_outside_0_to_360', (df.wind_direction_deg < 0) | (df.wind_direction_deg > 360), 'Direction should be within 0–360 degrees.'),
    ('humidity_outside_0_to_100', (df.humidity_pct < 0) | (df.humidity_pct > 100), 'Relative humidity should be within 0–100%.'),
    ('nonpositive_gearbox_oil_temperature', df.gearbox_oil_temp_C <= 0, 'Non-positive internal gearbox oil temperature is suspicious.'),
    ('nonpositive_gearbox_bearing_temperature', df.gearbox_bearing_temp_C <= 0, 'Non-positive internal bearing temperature is suspicious.'),
    ('nonpositive_generator_winding_temperature', df.generator_winding_temp_C <= 0, 'Non-positive internal winding temperature is suspicious.'),
    ('negative_vibration_measurement', (df.drivetrain_vibration_rms_mmps < 0) | (df.tower_vibration_mmps < 0), 'Vibration magnitude should not be negative.'),
    ('nonpositive_oil_pressure', df.oil_pressure_bar <= 0, 'Non-positive lubrication pressure is suspicious.'),
    ('negative_oil_particle_count', df.oil_particle_count < 0, 'Particle count should not be negative.'),
    ('negative_lifecycle_measurement', (df.operating_hours_total < 0) | (df.cumulative_energy_MWh < 0) | (df.load_cycles < 0) | (df.component_age_days < 0), 'Lifecycle totals should not be negative.'),
]
anomaly_summary = pd.DataFrame([{'screen': name, 'flagged_records': mask.fillna(False).sum(), 'flagged_percentage': mask.fillna(False).mean() * 100, 'review_reason': reason} for name, mask, reason in checks])
display(anomaly_summary.style.format({'flagged_percentage': '{:.3f}%'}))

extreme_columns = ['gearbox_oil_temp_C', 'gearbox_bearing_temp_C', 'generator_winding_temp_C', 'drivetrain_vibration_rms_mmps', 'tower_vibration_mmps', 'oil_particle_count', 'oil_pressure_bar']
extremes = df[extreme_columns].agg(['min', 'max', lambda x: x.quantile(.01), lambda x: x.quantile(.99)]).T
extremes.columns = ['minimum', 'maximum', 'p01', 'p99']
display(Markdown('### Extreme sensor-value context'))
display(extremes.round(3))
display(Markdown('**Finding:** The screens flag values that violate broad physical constraints. High-end values in temperatures, vibration, and oil particles may be genuine degradation signals rather than data errors; retain them for operational review.'))

## SCADA anomaly screens

,screen,flagged_records,flagged_percentage,review_reason
0,negative_power_output,0,0.000%,Power output below zero is physically implausible.
1,power_above_rated_power,13160,9.988%,Output above the recorded rated capacity warrants review.
2,negative_rotor_speed,0,0.000%,Negative rotational speed is implausible.
3,negative_generator_speed,0,0.000%,Negative rotational speed is implausible.
4,wind_speed_negative_or_above_40_mps,0,0.000%,Negative or exceptionally high wind-speed reading.
5,wind_direction_outside_0_to_360,0,0.000%,Direction should be within 0–360 degrees.
6,humidity_outside_0_to_100,0,0.000%,Relative humidity should be within 0–100%.
7,nonpositive_gearbox_oil_temperature,0,0.000%,Non-positive internal gearbox oil temperature is suspicious.
8,nonpositive_gearbox_bearing_temperature,0,0.000%,Non-positive internal bearing temperature is suspicious.
9,nonpositive_generator_winding_temperature,0,0.000%,Non-positive internal winding temperature is suspicious.


### Extreme sensor-value context

,minimum,maximum,p01,p99
gearbox_oil_temp_C,35.814,109.988,41.860,85.274
gearbox_bearing_temp_C,37.901,113.862,44.507,93.080
generator_winding_temp_C,44.771,158.996,50.749,133.448
drivetrain_vibration_rms_mmps,1.216,5.770,1.294,3.910
tower_vibration_mmps,0.801,3.717,0.828,2.039
oil_particle_count,50.565,305.105,51.265,217.853
oil_pressure_bar,3.836,6.689,4.916,6.311


**Finding:** The screens flag values that violate broad physical constraints. High-end values in temperatures, vibration, and oil particles may be genuine degradation signals rather than data errors; retain them for operational review.

In [6]:
# Target label quality
display(Markdown('## Target-variable quality checks'))
target_summary = df['failure'].value_counts(dropna=False).rename_axis('failure_value').reset_index(name='count')
target_summary['percentage'] = target_summary['count'].div(len(df)).mul(100)
display(target_summary.style.format({'percentage': '{:.3f}%'}))
missing_target = df.failure.isna().sum()
unexpected_target = (~df.failure.isin([0, 1]) & df.failure.notna()).sum()
display(pd.DataFrame({'missing_target_records': [missing_target], 'unexpected_nonbinary_target_records': [unexpected_target]}))
display(Markdown(f'**Finding:** The expected labels are 0 (normal) and 1 (fault). Missing labels: **{missing_target:,}**; unexpected non-binary labels: **{unexpected_target:,}**.'))

## Target-variable quality checks

,failure_value,count,percentage
0,0,127807,97.000%
1,1,3953,3.000%


,missing_target_records,unexpected_nonbinary_target_records
0,0,0


**Finding:** The expected labels are 0 (normal) and 1 (fault). Missing labels: **0**; unexpected non-binary labels: **0**.